# Global Settings

In [1]:
import random
import os

random.seed(2025)

input_path = os.path.join(os.path.join(os.curdir,'input'))
output_path = os.path.join(os.path.join(os.curdir,'output'))

In [2]:
import logging
from logging.handlers import RotatingFileHandler

# 创建一个日志记录器
logger = logging.getLogger(__name__)

# 设置日志的最低级别
logger.setLevel(logging.DEBUG)

# 创建控制台处理器（StreamHandler），将日志输出到控制台
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)  # 设置控制台日志级别

# 创建文件处理器（FileHandler），将日志输出到文件
file_handler = RotatingFileHandler('RoutePlanning.log', maxBytes=5*1024*1024, backupCount=3)  # 最大5MB，最多保留3个备份
file_handler.setLevel(logging.DEBUG)  # 设置文件日志级别

# 设置日志格式
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
console_handler.setFormatter(formatter)
file_handler.setFormatter(formatter)

# 将处理器添加到logger
logger.addHandler(console_handler)
logger.addHandler(file_handler)


# Load Data

In [3]:
import os
import geopandas as gpd

target_crs = 'EPSG:32649'

def load_shp(file_name):
    gdf = gpd.read_file(os.path.join(input_path, f"{file_name}.shp"))
    logger.info(f"Load data from {file_name}.shp ({gdf.crs})")
    if target_crs != gdf.crs:
        gdf = gdf.to_crs(target_crs)
    return gdf

### Static Env Data

In [4]:
spaces_file = 'space02'
map_file = 'map01'
buildings_file = 'building01'
rivers_file = 'river01'
roads_file = 'network01'
road_nodes_file = 'network_node01'

spaces = load_shp(spaces_file)
map_data = load_shp(map_file)
buildings = load_shp(buildings_file)
rivers = load_shp(rivers_file)
roads = load_shp(roads_file)
road_nodes = load_shp(road_nodes_file)

2025-02-16 00:09:31 - INFO - Load data from space02.shp (EPSG:32649)
2025-02-16 00:09:31 - INFO - Load data from map01.shp (EPSG:32649)
2025-02-16 00:09:32 - INFO - Load data from building01.shp (EPSG:32649)
2025-02-16 00:09:32 - INFO - Load data from river01.shp (EPSG:32649)
2025-02-16 00:09:32 - INFO - Load data from network01.shp (EPSG:32649)
2025-02-16 00:09:32 - INFO - Load data from network_node01.shp (EPSG:32649)


### Flood Data

In [5]:
floods_file = '720yy'
floods = load_shp(floods_file)

2025-02-16 00:09:32 - INFO - Load data from 720yy.shp (EPSG:32649)


### Shelter Data

In [6]:
shelters_file = 'shelter_plan01'
shelters = load_shp(shelters_file)

2025-02-16 00:09:32 - INFO - Load data from shelter_plan01.shp (EPSG:32649)


# Preprocessing

### Flood

In [7]:
def clip_flood(flood_data):
    floods_clipped = gpd.overlay(flood_data, rivers, how='difference')
    logger.debug(f"Flood area: {flood_data.geometry.area.sum()} -> {floods_clipped.geometry.area.sum()}")
    return floods_clipped

### Map

In [8]:
# 提取地图边界
map_bound = map_data.geometry.unary_union.envelope
logger.info(f"Map Boundary: {map_bound.bounds}")

/tmp/ipykernel_16409/4059485571.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  map_bound = map_data.geometry.unary_union.envelope
2025-02-16 00:09:35 - INFO - Map Boundary: (731708.1476547337, 3835016.5231311666, 753519.3080212934, 3859710.4085347075)


### Underground Space (Start)

In [9]:
# 提取spaces和shelters的坐标
spaces_coords = [(point.x, point.y) for point in spaces.geometry if point.is_valid]
logger.info(f"Number of Underground Spaces: {len(spaces_coords)}")

2025-02-16 00:09:35 - INFO - Number of Underground Spaces: 1283


In [10]:
space_radius = 100
_space_circles = []
for point in shelters.geometry:
    space_circle = point.buffer(space_radius)
    _space_circles.append(space_circle)
space_circles = gpd.GeoDataFrame(geometry=_space_circles)

### Shelter (End)

In [11]:
shelters_coords = [(point.x, point.y) for point in shelters.geometry if point.is_valid]
logger.info(f"Number of Shelters: {len(shelters_coords)}")

2025-02-16 00:09:36 - INFO - Number of Shelters: 171


In [18]:
def build_shelter_circles(distance_threshold=100):
    _shelter_circles = []
    # 遍历每个避难所点，创建圆形并检查与河流的相交关系
    for point in shelters.geometry:
        # 创建避难所的圆形（buffer）
        shelter_circle = point.buffer(distance_threshold)
        
        # 检查圆形是否与河流相交
        if any(shelter_circle.intersects(river) for river in rivers.geometry):
            # 如果与任意河流相交，则跳过该圆形
            continue            
        _shelter_circles.append(shelter_circle)

    # 将圆形数据转为GeoDataFrame
    shelter_circles = gpd.GeoDataFrame(geometry=_shelter_circles)
    return shelter_circles

In [19]:
# shelter_circles = build_shelter_circles(distance_threshold=100)

# Functions

### Distance Calculation

In [20]:
# 计算两个点之间的欧几里得距离
def euclidean_distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

### Shape in map

In [21]:
import numpy as np
import geopandas as gpd
import random
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, LineString
import heapq


def step_condition(start, end):
    line = LineString([start, end])

    if not line.within(map_data.geometry.unary_union):
        return "map", map_data.geometry.unary_union

    for flood in floods.geometry:
        if flood.intersects(line):  # 判断路径段是否与积水相交
            return "flood", flood

    for shelter in shelter_circles.geometry:
        if shelter.intersects(line):  # 判断路径段是否与避难所相交
            return "shelter", shelter

    for river in rivers.geometry:
        if river.intersects(line):  # 判断路径段是否与河流相交
            return "river", river

    for building in buildings.geometry:
        if building.intersects(line):  # 判断路径段是否与建筑物相交
            return "building", building

    # 如果路径段没有经过障碍物或避难所
    return "pass", None

def is_connected(x, y):
    condition, obj = step_condition(x, y)
    if condition not in ["map", "river", "flood"]:
        return True
    return False

# 判断点是否在障碍物中
def is_in_obstacle(x, y):
    point = Point(x, y)

    if not map_data.geometry.unary_union.contains(point):
        logger.debug(f"Point({x},{y}) not in the map")
        return True

    for geom in floods.geometry:
        if geom.contains(point):
            return True

    for geom in rivers.geometry:
        if geom.contains(point):
            logger.debug(f"Point({x},{y}) in a river")
            return True

    return False

# 生成随机点
def generate_random_point(boundary):
    xmin, ymin, xmax, ymax = boundary
    while True:
        x = random.uniform(xmin, xmax)
        y = random.uniform(ymin, ymax)
        if not is_in_obstacle(x, y):  # 确保点不在障碍物中
            return (x, y)


### Visualization

In [22]:
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import os
import random
from matplotlib import cm


# 可视化路径和地图的函数
def visualize_routes(routes, saveFig=False, saveShp=False, **kwargs):
    logger.info(f"Plotting ({len(routes)} routes) ... ")
    # 绘制地图边界
    fig, ax = plt.subplots(figsize=(10, 10))
    map_data.boundary.plot(ax=ax, color='black', linewidth=1)
    map_data.plot(ax=ax, color='gray', alpha=0.6)

    # 绘制障碍物
    buildings.plot(ax=ax, color='brown', alpha=0.6, label='Buildings')
    rivers.plot(ax=ax, color='blue', alpha=0.4, label='Rivers')
    draw_floods = kwargs.get('draw_floods', False)
    if draw_floods:
        floods.plot(ax=ax, color='purple', alpha=0.4, label='Floods')

    # 使用colormap设置路径颜色
    colormap = cm.viridis  # 使用 viridis colormap
    ends_set = set([route[-1] for route in routes])  # 终点数量
    ends_colors = [colormap(i / len(ends_set)) for i in range(len(ends_set))]  # 为不同终点路径分配不同的颜色
    route_colors = dict(zip(ends_set, ends_colors))
    # 绘制路径
    for route in routes:
        if route:
            route_x, route_y = zip(*route)
            ax.plot(route_x, route_y, color=route_colors[route[-1]], label='Route', linewidth=1, zorder=4)

            # 绘制起点和终点
            ax.scatter(route[0][0], route[0][1], color='red', label='Start', zorder=6, s=10)
            ax.scatter(route[-1][0], route[-1][1], color='green', label='End', zorder=6, s=10)

    # 图例不重复
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[1:4], labels[1:4], loc='upper left')

    ax.set_title('Routes Visualization')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True)
    plt.axis('equal')


    # 保存图片
    if len(routes) == 1:
        output_name = f"{route[0][-1]}"
    else:
        output_name = f"r{len(routes)}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
    if saveFig:
        output_file = os.path.join(os.path.join(os.curdir,'output'), f"{output_name}.jpg")
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        logger.info(f"Save Fig {output_file}")
    if saveShp:
        output_file = os.path.join(os.path.join(os.curdir,'output'), f"{output_name}.shp")
        # 创建一个GeoDataFrame来存储路径
        lines = []
        for route in routes:
            if route and len(route)>1:
                # 创建LineString对象
                line = LineString(route)
                lines.append(line)
        # 将LineString对象存入GeoDataFrame
        gdf = gpd.GeoDataFrame(geometry=lines)
        gdf.set_crs(target_crs, allow_override=True, inplace=True)

        # 保存为Shapefile
        gdf.to_file(output_file)
        logger.info(f"Save Fig {output_file}")

    plt.show()

In [23]:
import networkx as nx
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, LineString


def plot_graph(G, **kwargs):
    # 绘制地图边界
    fig, ax = plt.subplots(figsize=(10, 10))
    map_data.boundary.plot(ax=ax, color='black', linewidth=1)
    map_data.plot(ax=ax, color='gray', alpha=0.1)

    pos = nx.get_node_attributes(G, "pos")  # 获取节点位置
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="gray", alpha=0.3, width=0.5)
    # nx.draw_networkx_nodes(G, pos, ax=ax, node_size=1, node_color="blue", alpha=0.8)

    # 绘制其他环境信息
    draw_buildings = kwargs.get('draw_buildings', buildings)
    draw_rivers = kwargs.get('draw_rivers', rivers)
    draw_floods = kwargs.get('draw_floods', None)
    if draw_buildings is not None:
        draw_buildings.plot(ax=ax, color='brown', alpha=0.6, label='Buildings')
    if draw_rivers is not None:
        draw_rivers.plot(ax=ax, color='blue', alpha=0.4, label='Rivers')
    if draw_floods is not None:
        draw_floods.plot(ax=ax, color='purple', alpha=0.4, label='Floods')

    # 绘制路线
    routes = kwargs.get('routes', None)
    if routes:
        # 使用colormap设置路径颜色
        colormap = cm.viridis  # 使用 viridis colormap
        ends_set = set([route[-1] for route in routes])  # 终点数量
        ends_colors = [colormap(i / len(ends_set)) for i in range(len(ends_set))]  # 为不同终点路径分配不同的颜色
        route_colors = dict(zip(ends_set, ends_colors))
        # 绘制路径
        for route in routes:
            if route:
                if len(route)==1:
                    ax.scatter(route[0][0], route[0][1], color='red', label='Start', alpha=0.4, zorder=6, s=10)
                    continue
                route_x, route_y = zip(*route)
                ax.plot(route_x, route_y, color=route_colors[route[-1]], label='Route', linewidth=1, zorder=4)

                # 绘制起点和终点
                ax.scatter(route[0][0], route[0][1], color='red', label='Start', alpha=0.4, zorder=6, s=10)
                ax.scatter(route[-1][0], route[-1][1], color='green', label='End', alpha=0.4, zorder=6, s=10)

    # ax.set_title('Routes Visualization')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True)
    plt.axis('equal')

    saveFig = kwargs.get('saveFig', False)
    output_name = kwargs.get('file_name', datetime.now().strftime('%Y%m%d%H%M%S'))
    if saveFig:
        output_file = os.path.join(os.path.join(os.curdir,'output'), f"{output_name}.jpg")
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        logger.info(f"Save Fig {output_file}")

    plt.show()

###  Save & Load Routes

In [24]:
from datetime import datetime
import os
import re

def save_routes(start_coords, route, file_path, show=False):
    with open(file_path, "a") as file:
        if route:
            if show:
                visualize_routes([route])
            logger.debug(f"Succeed from {start_coords} to {route[-1]} \n")
            file.write(",".join(map(str, route)) + "\n")
        else:
            logger.debug(f"Failed at {start_coords}\n")
            file.write(f"No valid path found for start point {start_coords} \n")


def load_routes(file_path):
    logger.debug(f"Read routes from {file_path}")
    paths = []
    with open(file_path, "r") as file:
        for line in file:
            line = line.strip()
            line = line.replace('[', '(').replace(']', ')')  # 替换方括号为圆括号
            # 跳过不需要的行
            if not line:
                continue
            if line.startswith("No valid path"):
                # logger.debug(f"{line}")
                continue
            # 如果路径有效，将每个坐标点拆分并转换为数字
            try:
                points = re.findall(r'\((-?\d*\.?\d+),\s*(-?\d*\.?\d+)\)', line)
                route = [(float(x), float(y)) for x, y in points]
                # 添加有效的路径
                if route:
                    # logger.debug(f"Extract Route: {route}")
                    paths.append(route)
            except Exception as e:
                logger.error(f"Error processing line: {line}. Error: {e}")
    return paths

In [25]:
import pandas as pd

def convert_coords_to_index(coords, coords_list):
    if coords in coords_list:
        return coords_list.index(coords)
    else:
        return None  # 如果找不到该坐标点，可以返回 None


def parse_routes_to_dataframe(routes, starts_coords, ends_coords):
    """
    解析路径数据为 DataFrame 格式，并计算每条路径的距离。

    :param routes: list, 每个路径由坐标点组成的列表
    :param starts_coords: 起点坐标列表，格式为[(lon1, lat1), (lon2, lat2), ...]
    :param ends_coords: 终点坐标列表，格式为[(lon1, lat1), (lon2, lat2), ...]
    :return: pandas DataFrame, 包含路径信息及其距离
    """
    data = []

    for route in routes:
        # 计算路径的总距离
        route_distance = 0
        for j in range(len(path) - 1):
            route_distance += euclidean_distance(path[j], path[j + 1])

        # 添加数据到列表
        data.append({
            "start_coords": route[0],
            "end_coords": route[-1],
            "path": route,  # 原始路径坐标
            "distance": route_distance
        })

    # 创建 DataFrame
    df = pd.DataFrame(data)
    missing_coords = [coord for coord in starts_coords if coord not in df['start_coords'].values]
    missing_df = pd.DataFrame({'start_coords': missing_coords})
    df = pd.concat([df, missing_df], ignore_index=True)

    df['start_index'] = df['start_coords'].apply(lambda x: convert_coords_to_index(x, starts_coords))
    df['end_index'] = df['end_coords'].apply(lambda x: convert_coords_to_index(x, ends_coords))


    return df


# Search Algorithm

### Build Graph

In [26]:
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

def build_road_graph(edges, nodes):
    # 创建 NetworkX 图
    G = nx.Graph()

    # 创建节点坐标字典
    node_dict = { (node.geometry.x, node.geometry.y): node["OBJECTID"] for _, node in nodes.iterrows() }

    # 添加节点
    for _, node in nodes.iterrows():
        G.add_node(node["OBJECTID"], pos=(node.geometry.x, node.geometry.y))

    # 添加边
    for _, edge in edges.iterrows():
        start_coord = edge.geometry.coords[0]  # 线的起点 (x, y)
        end_coord = edge.geometry.coords[-1]   # 线的终点 (x, y)
        if start_coord == end_coord:
            continue

        # 查找节点 ID
        start_id = node_dict.get(start_coord, None)
        end_id = node_dict.get(end_coord, None)

        # 如果找不到确切匹配的节点，则尝试最近匹配
        if start_id is None:
            start_id = min(node_dict, key=lambda k: Point(k).distance(Point(start_coord)))
            start_id = node_dict[start_id]

        if end_id is None:
            end_id = min(node_dict, key=lambda k: Point(k).distance(Point(end_coord)))
            end_id = node_dict[end_id]

        # 添加边
        G.add_edge(start_id, end_id, weight=edge["Shape_Leng"])  # 使用 Shape_Leng 作为权重

    return G

In [27]:
# graph = build_road_graph(edges=roads, nodes=road_nodes)
# logger.info(f"Build graph with {len(graph.nodes)} nodes and {len(graph.edges)} edges")

In [28]:
# # 保存为 GML 文件
# nx.write_gml(graph, os.path.join(output_path, "graph.gml"))

### Load Graph

In [29]:
import networkx as nx
import os
import ast

def load_graph(file_name):
    # 加载 GML 文件中的图
    graph = nx.read_gml(os.path.join(output_path, f"{file_name}.gml"))

    # 创建新的空图
    new_graph = nx.Graph()

    # 重新添加节点
    for node, attributes in graph.nodes(data=True):
        new_node = ast.literal_eval(node)  # 解析字符串格式的元组
        new_graph.add_node(new_node, **attributes)

    # 重新添加边
    for u, v, attributes in graph.edges(data=True):
        new_u = ast.literal_eval(u)
        new_v = ast.literal_eval(v)
        new_graph.add_edge(new_u, new_v, **attributes)

    logger.debug(f"Load graph with {len(new_graph.nodes)} nodes and {len(new_graph.edges)} edges")
    return new_graph

In [30]:
# # 替换原始 graph
# graph = load_graph(file_name='graph_0')

### Graph with Flood

In [31]:
import networkx as nx
import geopandas as gpd
from shapely.geometry import LineString

def remove_flooded_edges(G, floods):
    """
    删除与洪水区域相交的所有边
    :param G: networkx Graph, 包含道路的图
    :param floods: GeoDataFrame, 包含洪水区域的多边形
    :return: 修改后的 networkx Graph
    """
    edges_to_remove = []

    # 遍历所有边，检查是否与洪水区域相交
    for u, v, data in G.edges(data=True):
        if "geometry" in data:  # 确保边有几何信息
            edge_geom = data["geometry"]
        else:
            # 如果没有geometry信息，尝试用节点坐标生成 LineString
            pos = nx.get_node_attributes(G, "pos")
            edge_geom = LineString([pos[u], pos[v]])

        # 判断边是否与洪水区域相交
        if floods.geometry.intersects(edge_geom).any():
            edges_to_remove.append((u, v))

    # 从图中删除受影响的边
    G.remove_edges_from(edges_to_remove)

    logger.info(f"Remove {len(edges_to_remove)} edges")
    return G


In [32]:
# graph_flooded = remove_flooded_edges(graph, floods)
# logger.info(f"Build flooded graph has {len(graph_flooded.nodes)} nodes and {len(graph_flooded.edges)} edges")
# nx.write_gml(graph_flooded, os.path.join(output_path, "graph_03yy.gml"))
# plot_graph(G=graph_flooded, draw_floods=True)

#### RUN - All in One

In [33]:
# graph = load_graph(file_name='graph_0')

# for flood_tag in ['03', '05', '10', '30', '50', '100', '720']:
#     floods = load_shp(f"{flood_tag}yy01") if flood_tag!='720' else load_shp(f"{flood_tag}yy")
#     floods = clip_flood(floods)
        
#     graph_flooded = remove_flooded_edges(graph, floods)
#     logger.info(f"Build flooded graph has {len(graph_flooded.nodes)} nodes and {len(graph_flooded.edges)} edges")
#     nx.write_gml(graph_flooded, os.path.join(output_path, f"graph_{flood_tag}.gml"))

#     plot_graph(G=graph_flooded, draw_floods=floods)

### Dijkstra Algorithm

In [34]:
import networkx as nx

def plan_Dijkstra(G, start, end):
    """
    计算最短路径，并返回最短路径及其权重
    :param G: networkx 图
    :param start: 起点坐标 (x, y) 或节点 ID
    :param end: 终点坐标 (x, y) 或节点 ID
    :return: tuple, (shortest_path, path_weight)
    """
    try:
        # 计算最短路径
        path = nx.shortest_path(G, source=start, target=end, weight="weight")
        path_weight = nx.path_weight(G, path, weight="weight")

        return path, path_weight

    except nx.NetworkXNoPath:
        # 如果没有路径
        return None, float("inf")


In [35]:
def convert_path_to_coordinates(G, path, start_coords=None, end_coords=None):
    """
    将路径中的节点ID转换为坐标点，并插入起点和终点的坐标
    :param G: networkx 图
    :param path: 由节点ID组成的路径
    :param start_coords: 起点坐标，若为 None，则不插入
    :param end_coords: 终点坐标，若为 None，则不插入
    :return: 路径对应的坐标点列表
    """
    if path is None or len(path)==0:
        return None
    coordinates = []
    # 如果有起点坐标，则先加入起点坐标
    if start_coords:
        coordinates.append(start_coords)
    # 遍历路径中的每个节点，获取其坐标
    for node in path:
        if node in G.nodes:
            coordinates.append(G.nodes[node].get('pos'))  # 假设坐标保存在 'pos' 属性中
    # 如果有终点坐标，则加入终点坐标
    if end_coords:
        coordinates.append(end_coords)
    return coordinates

In [36]:
# 找到最近的 shelter，且距离超过阈值的忽略
def find_shelters_nearby(coords, distance_threshold):
    """
    根据起点坐标找到最近的 shelter，且距离超过阈值的忽略
    :param coords: 起点坐标 (x, y)
    :param distance_threshold: 距离阈值，超过该距离的 shelter 被忽略
    :return: 最近的 shelter 坐标或 None
    """
    min_distance = float('inf')
    nearest_shelter = None

    for shelter_coords in shelters_coords:
        # 计算起点与 shelter 的欧几里得距离
        distance = euclidean_distance(coords, shelter_coords)
        
        # 如果距离超过阈值，则忽略该 shelter
        if distance > distance_threshold:
            continue
        
        # 更新最近的 shelter
        if distance < min_distance:
            min_distance = distance
            nearest_shelter = shelter

    return nearest_shelter

In [39]:
import networkx as nx

def find_nodes_nearby(G, coords, distance_threshold, ignore_flood=True):
    """
    找到给定坐标周围一定距离内的图节点
    :param G: networkx 图
    :param coords: 给定坐标 (x, y)
    :param distance_threshold: 距离阈值，单位与坐标一致
    :return: 一个包含附近节点ID的列表
    """
    nearby_nodes = []

    # 遍历所有节点
    for node, data in G.nodes(data=True):
        node_coords = data.get('pos')  # 假设节点的坐标保存在 'pos' 属性中
        if node_coords is None:
            continue

        # 计算节点和目标坐标的欧几里得距离
        distance = euclidean_distance(coords, node_coords)  # 计算两点之间的距离，单位：米
        if distance <= distance_threshold:  # 如果距离小于或等于阈值，则认为该节点在附近
            if not ignore_flood:    # 如果积水导致不联通，跳过
                if not is_connected(coords, node_coords):
                    continue
            nearby_nodes.append(node)

    return nearby_nodes


### RUN

In [ ]:
shelter_tag = "plan"
for flood_tag in ['0', '03', '05', '10', '30', '50', '100', '720']:
    logger.info(f"RUN flood:{flood_tag} shelter:{shelter_tag}")
    if flood_tag != '0':
        floods_file = f'{flood_tag}yy01' if flood_tag!='720' else f'{flood_tag}yy'
        floods = load_shp(floods_file)
        floods['intersection'] = floods.geometry.apply(lambda x: rivers.geometry.intersects(x).any())
        floods = floods[~floods['intersection']].drop(columns=['intersection'])
        logger.debug(f"Flood area (remove rivers): {floods.geometry.area.sum()}")

    graph_flooded = load_graph(file_name=f'graph_{flood_tag}')

    route_file = os.path.join(os.curdir, "output", f"routes_Dijkstra_{flood_tag}_{shelter_tag}.txt")
    paths = dict()
    for space_coords in spaces_coords:
        nearest_shelter = find_shelters_nearby(coords=space_coords, distance_threshold=100)
        if nearest_shelter:
            save_routes(start_coords=space_coords, route=[space_coords, nearest_shelter], file_path=route_file, show=False)
            continue
        start_nodes = find_nodes_nearby(G=graph_flooded, coords=space_coords, distance_threshold=100, ignore_flood=True)
        best_route = None
        best_route_weight = float('inf')
        best_shelter = None
        for start in start_nodes:
            for shelter_coords in shelters_coords:
                end_nodes = find_nodes_nearby(G=graph_flooded, coords=shelter_coords, distance_threshold=100, ignore_flood=True)
                for end in end_nodes:
                    if (start, end) in paths.keys():
                        path_weight = paths[(start, end)]['weight']
                        if best_route_weight > path_weight:
                            best_route_weight = path_weight
                            best_route = paths[(start, end)]['path']
                            best_shelter = paths[(start, end)]['shelter']
                        continue
                    path, path_weight = plan_Dijkstra(G=graph_flooded, start=start, end=end)
                    paths[(start, end)] = {'path': path, 'weight': path_weight, 'shelter': shelter_coords}
                    if best_route_weight > path_weight:
                        best_route_weight = path_weight
                        best_route = path
                        best_shelter = shelter_coords
        route = convert_path_to_coordinates(G=graph_flooded, path=best_route, start_coords=space_coords, end_coords=best_shelter)
        save_routes(start_coords=space_coords, route=route, file_path=route_file, show=False)

2025-02-16 00:12:26 - INFO - RUN flood:0 shelter:plan


# Sample-Based Algorithm

### RRT

In [ ]:
def plan_RRT(start_coords, **kwargs):
    """
    RRT路径规划，目标是到达多个可能的终点中的任意一个附近
    :param start_coords: 起始坐标
    :param kwargs: 其他参数，如步长、最大迭代次数等
    :return: 路径列表
    """
    logger.info(f"RRT: {start_coords}")
    start = np.array(start_coords)
    nodes = [start]
    parent = {tuple(start): None}
    route_found = False
    all_routes = []

    step_size = kwargs.get('step_size', 50)  # 步长
    max_iter = kwargs.get('max_iter', 1000)  # 最大迭代次数
    # boundary = kwargs.get('boundary', (start[0]-2500, start[1]-2500, start[0]+2500, start[1]+2500))  # 周围2.5km
    boundary = kwargs.get('boundary', map_bound.bounds)

    debug = kwargs.get('debug', False)
    if debug:
        file_path=datetime.now().strftime('%Y%m%d%H%M%S')

    for i in range(max_iter):
        # 随机生成一个点
        rand_point = generate_random_point(boundary)

        # 找到距离随机点最近的节点
        nearest_node = min(nodes, key=lambda node: euclidean_distance(node, rand_point))

        # 计算从最近节点到随机点的单位向量
        direction = np.array(rand_point) - np.array(nearest_node)
        distance = np.linalg.norm(direction)
        direction /= distance  # 单位化

        # 如果距离太远，则按步长限制扩展的长度
        step = min(step_size, distance)
        new_node = np.array(nearest_node) + direction * step

        # 判断路径段是否经过障碍物或避难所
        condition, obj = step_condition(nearest_node, new_node)
        if condition in ["river", "flood", "map"]: # building穿过去也合理，太多building了
            logger.debug(f"{condition} at {nearest_node} -> {new_node}")
            continue

        if condition == "shelter":
            new_node = (obj.centroid.x, obj.centroid.y)
            logger.debug(f"Arrived at shelter: {condition} at {nearest_node} -> {new_node}")
            route_found = True
            end = new_node

            # 将新节点加入到树中
            nodes.append(tuple(new_node))
            parent[tuple(new_node)] = tuple(nearest_node)
            logger.debug(f"Adding new_node {tuple(new_node)} with parent {tuple(nearest_node)}")

            if debug:
                # 保存所有路径
                all_routes.append([nearest_node, new_node])
                lines = [LineString(route) for route in all_routes]

                try:
                    visualize_route_tree(lines, saveFig=True, saveShp=False, file_path=file_path)
                except Exception as e:
                    logger.info(f"Error in visualizing route: {e}")

            break


        # 将新节点加入到树中
        nodes.append(tuple(new_node))
        parent[tuple(new_node)] = tuple(nearest_node)
        logger.debug(f"Adding new_node {tuple(new_node)} with parent {tuple(nearest_node)}")

        if debug:
            # 保存所有路径
            all_routes.append([nearest_node, new_node])
            lines = [LineString(route) for route in all_routes]
            if i%100 == 0:
                try:
                    visualize_route_tree(lines, saveFig=True, saveShp=False, file_path=file_path)
                except Exception as e:
                    logger.info(f"Error in visualizing route: {e}")

    # 如果找到了路径，回溯路径
    if route_found:
        route = [end]
        current_node = end
        while parent[tuple(current_node)]:
            current_node = parent[tuple(current_node)]
            route.append(current_node)
        route.reverse()
        return route
    else:
        return None


### Visualize RRT Tree

In [ ]:
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import os
import random
from matplotlib import cm


# 可视化路径和地图的函数
def visualize_route_tree(route, saveFig=False, saveShp=False, file_path="route"):
    logger.info(f"Plotting route with {len(route)} lines ... ")
    # 绘制地图边界
    fig, ax = plt.subplots(figsize=(10, 10))
    map_data.boundary.plot(ax=ax, color='black', linewidth=1, label='City Boundary')
    map_data.plot(ax=ax, color='gray', alpha=0.6, label='City')

    # 绘制障碍物
    buildings.plot(ax=ax, color='brown', alpha=0.6, label='Building')
    rivers.plot(ax=ax, color='blue', alpha=0.4, label='River')

    shelters.plot(ax=ax, color='green', alpha=0.6, markersize=shelter_radius, label='Shelters')

    # 绘制路径
    for line in route:
        ax.plot(*line.xy, color='yellow', linewidth=2)

    ax.set_title('Routes Visualization')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True)
    plt.axis('equal')

    # 保存图片
    output_name = f"l{len(route)}"

    if saveFig or saveShp:
        os.makedirs(os.path.join(os.curdir,'output',file_path), exist_ok=True)
    if saveFig:
        output_file = os.path.join(os.curdir,'output',file_path, f"{output_name}.jpg")
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        logger.info(f"Save Fig {output_file}")
    if saveShp:
        output_file = os.path.join(os.curdir,'output',file_path, f"{output_name}.shp")
        # 创建一个GeoDataFrame来存储路径
        lines = []
        for route in routes:
            # 创建LineString对象
            line = LineString(route)
            lines.append(line)
        # 将LineString对象存入GeoDataFrame
        gdf = gpd.GeoDataFrame(geometry=lines)
        gdf.set_crs(target_crs, allow_override=True, inplace=True)

        # 保存为Shapefile
        gdf.to_file(output_file)
        logger.info(f"Save Fig {output_file}")

    # plt.show()

### RUN - Route Planning: RRT

In [ ]:
params = {
    "step_size": 100,
    "max_iter": 3000,
    "debug": True,
}
logger.debug(f"Params: {params}")

route_file = os.path.join(output_path, f"routes_RRT.txt")
for start in spaces_coords:
    route = plan_RRT(start_coords=start, **params)
    save_routes(start_coords=start, route=route, file_path=route_file)

# Route Analysis

In [ ]:
shelter_tag = "plan"
flood_tag = '0'

route_file = f"routes_Dijkstra_{flood_tag}_{shelter_tag}.txt"
routes = load_routes(file_path=os.path.join(output_path,route_file))
graph = load_graph(f"graph_{flood_tag}")
plot_graph(G=graph, routes=routes, draw_floods=True,
    saveFig=True, file_name=f"routes_Dijkstra_{flood_tag}_{shelter_tag}")

In [ ]:
# 1. 所有spaces加成灰色小点
#
# 2. 按照长度设置阈值：top[10：800；30：2000；60：5000]btn -> 计算平均值
# 只画路网：路网灰色+四种长度
# 画所有的shelter（最上），space（最下）
#
# 3. 密度：颜色深浅：同色
# 只画路网：路网灰色
# 画线段，不断叠加（透明度0.5）
# 画所有的shelter（最上），space（最下）

# Evaluation Index

### 1. Evacuation Related Index

**疏散时间（Evacuation Time）**
   - **定义：** 从灾难开始到所有人员安全撤离所需的总时间。
   - **评价标准：**
     - 较短的疏散时间是优化疏散算法的核心目标之一。在洪灾等紧急情况下，时间越短越能有效减少伤亡。
     - 可以通过模拟不同疏散路径的总时间来评估不同算法的效率。

**路径长度（Path Length）**
   - **定义：** 从地下空间到逃生点的总疏散路径的长度。
   - **评价标准：**
     - 路径长度越短，人员疏散的效率越高，能够减少时间和能源消耗。
     - 路径的设计应考虑障碍物、地形、水位等因素，避免冗长的绕行。

**安全性（Safety）**
   - **定义：** 疏散过程中每个疏散路径的安全性，包括避开洪水区域、保持通行畅通等。
   - **评价标准：**
     - 安全性是最重要的标准，尤其是在动态环境（如洪水、火灾等）下。应确保路径规划算法能够有效避开水位较高或障碍较多的区域。
     - 可以通过模拟不同路径在洪灾期间的风险水平（如水流、障碍物的阻挡等）来评估。

**拥堵度（Congestion Level）**
   - **定义：** 疏散路径上人员的密集程度，通常通过每条路径上的人员流量和每个节点的通行能力来衡量。
   - **评价标准：**
     - 在大规模疏散过程中，过于拥堵的路径会导致疏散延误和伤亡增加。疏散算法需要尽量避免路径拥堵，或者采取分流策略。
     - 可以通过模拟每个路径上的流量和拥堵情况来对算法进行评价。

### 2. Model Related Index

**计算复杂度（Computational Complexity）**
   - **定义：** 路径规划算法所需的计算资源和时间。
   - **评价标准：**
     - 高效的算法应能快速提供疏散路径，特别是在灾难发生时，实时性非常重要。
     - 评估不同算法的计算时间和资源消耗，确保算法能在有限的时间内提供有效路径。

**适应性（Adaptability）**
   - **定义：** 疏散算法在动态环境（如洪水水位变化、障碍物移动等）中的应变能力。
   - **评价标准：**
     - 在洪灾等变化快速的环境中，疏散算法需要具备实时更新和调整路径的能力，以适应水位变化、道路封闭等因素。
     - 可通过模拟不同的动态变化情况，评估算法的灵活性和适应性。
**鲁棒性（Robustness）**
   - **定义：** 算法在不同环境和条件下的稳定性。
   - **评价标准：**
     - 疏散路径规划应具有较高的鲁棒性，能够处理各种不确定性（如传感器误差、环境变化等）。
     - 通过在多个模拟场景下测试算法，评估其鲁棒性。

**扩展性（Scalability）**
   - **定义：** 算法能否适应不同规模的疏散任务。
   - **评价标准：**
     - 对于大规模城市的疏散问题，算法应能处理大量的人员和复杂的地下空间环境。可以通过模拟不同规模的疏散任务，评估算法的扩展性。
